In [1]:
import kagglehub
import pandas as pd
import seaborn as sns
from pathlib import Path
import plotly.graph_objects as go

In [2]:
# Load datasets
path = kagglehub.dataset_download("anikannal/cms-synthetic-data")

# Read in datasets
datasets = {
    fp.stem: pd.read_csv(fp)
    for fp in Path(path).glob("*.csv")
}

C:\Users\steph\AppData\Local\Temp\ipykernel_8936\3649949676.py:6: DtypeWarning: Columns (21: ICD9_DGNS_CD_10, 23: ICD9_PRCDR_CD_2, 24: ICD9_PRCDR_CD_3, 25: ICD9_PRCDR_CD_4, 26: ICD9_PRCDR_CD_5, 27: ICD9_PRCDR_CD_6) have mixed types. Specify dtype option on import or set low_memory=False.
  fp.stem: pd.read_csv(fp)


In [3]:
# Extract general information
def overview(datasets: dict):
    """Extract general overview information from a collection of datasets"""
    overview = {
        k:[]
        for k in ["dataset", "shape", "unique_rows"]
    }
    for name,df in datasets.items():
        overview["dataset"].append(name)
        overview["shape"].append(df.shape)
        overview["unique_rows"].append(len(df.drop_duplicates()))
    return pd.DataFrame(overview)

dataset_overview = overview(datasets)
dataset_overview

,dataset,shape,unique_rows
0,DE1_0_2008_Beneficiary_Summary_File_Sample_1,"(116352, 32)",116352
1,DE1_0_2008_to_2010_Inpatient_Claims_Sample_1,"(66773, 81)",66773
2,DE1_0_2008_to_2010_Outpatient_Claims_Sample_1,"(790790, 76)",790790
3,DE1_0_2009_Beneficiary_Summary_File_Sample_1,"(114538, 32)",114538
4,DE1_0_2010_Beneficiary_Summary_File_Sample_1,"(112754, 32)",112754


In [4]:
# Combine similar datasets
datasets["beneficiary_summary"] = pd.concat([d for n,d in datasets.items() if "Beneficiary_Summary_File" in n])
print(f"New shape: {datasets["beneficiary_summary"].shape}")

# Filter down datasets
datasets = {k:v for k,v in datasets.items() if "Beneficiary_Summary_File" not in k}
dataset_overview = overview(datasets)
dataset_overview

New shape: (343644, 32)


,dataset,shape,unique_rows
0,DE1_0_2008_to_2010_Inpatient_Claims_Sample_1,"(66773, 81)",66773
1,DE1_0_2008_to_2010_Outpatient_Claims_Sample_1,"(790790, 76)",790790
2,beneficiary_summary,"(343644, 32)",322128


In [5]:
# Attributes
attributes = {}
for name,df in datasets.items():
    attr_data = {
        k:[]
        for k in ["attribute", "dtype", "empty_vals"]
    }
    for col in df.columns:
        attr_data["attribute"].append(col)
        attr_data["dtype"].append(df[col].dtype)
        attr_data["empty_vals"].append(sum(df[col].isna()))
    attr_df = pd.DataFrame(attr_data)
    attributes[name] = attr_df
    
    print(name)
    display(attr_df)

DE1_0_2008_to_2010_Inpatient_Claims_Sample_1


,attribute,dtype,empty_vals
0,DESYNPUF_ID,str,0
1,CLM_ID,int64,0
2,SEGMENT,int64,0
3,CLM_FROM_DT,float64,68
4,CLM_THRU_DT,float64,68
...,...,...,...
76,HCPCS_CD_41,float64,66773
77,HCPCS_CD_42,float64,66773
78,HCPCS_CD_43,float64,66773
79,HCPCS_CD_44,float64,66773


DE1_0_2008_to_2010_Outpatient_Claims_Sample_1


,attribute,dtype,empty_vals
0,DESYNPUF_ID,str,0
1,CLM_ID,int64,0
2,SEGMENT,int64,0
3,CLM_FROM_DT,float64,11253
4,CLM_THRU_DT,float64,11253
...,...,...,...
71,HCPCS_CD_41,str,778122
72,HCPCS_CD_42,str,778914
73,HCPCS_CD_43,str,779594
74,HCPCS_CD_44,str,780234


beneficiary_summary


,attribute,dtype,empty_vals
0,DESYNPUF_ID,str,0
1,BENE_BIRTH_DT,int64,0
2,BENE_DEATH_DT,float64,338183
3,BENE_SEX_IDENT_CD,int64,0
4,BENE_RACE_CD,int64,0
5,BENE_ESRD_IND,str,0
6,SP_STATE_CODE,int64,0
7,BENE_COUNTY_CD,int64,0
8,BENE_HI_CVRAGE_TOT_MONS,int64,0
9,BENE_SMI_CVRAGE_TOT_MONS,int64,0


In [9]:
for name,df in datasets.items():
    for col in df.columns:
        if df[col].dtype in ["int64", "float64"]:
            fig = go.Figure(
                data = [
                    go.Histogram(
                        x = df[col],
                    )
                ],
                layout = {
                    "title": f"{name}: {col}",
                    "xaxis_title": col,
                }
            )
            Path(f"plots/{name}").mkdir(parents=True, exist_ok=True)
            fig.write_html(f"plots/{name}/{col}.html")